# Lab 11 — Reinforcement Learning with GRPO

**Goal:** improve Qwen's one-step Wordle decisions using reinforcement learning and the frozen Lab 10 reward.

This lab changes the training loop fundamentally.

Supervised learning:

```text
prompt → known target → cross-entropy
```

GRPO:

```text
prompt
  ↓
current policy samples several guesses
  ↓
Wordle environment scores each guess
  ↓
compare rewards within the group
  ↓
increase probability of better guesses
  ↓
repeat with the updated policy
```

The data is now **on-policy**: the training signal comes from outputs generated by the current model during training.

We are not trying to perfect Wordle. We are learning how online RL differs from SFT, LoRA, and distillation.

## 11.1 Why GRPO?

GRPO generates multiple completions for the same prompt and computes an advantage from their relative rewards.

For a group of four guesses:

```text
CRANE  reward +0.05
SLATE  reward +0.37
PLANT  reward +2.80
xxxxx  reward -0.25
```

the group mean becomes the baseline. `PLANT` receives a strongly positive advantage; malformed or weak actions receive negative advantage.

Unlike PPO, GRPO does not require a separately trained value model.

Current TRL supports:

- online GRPO generation;
- custom Python reward functions;
- PEFT/LoRA adapters;
- conversational prompts;
- group-relative reward scaling.

We will use all four.

## 11.2 Experimental choices

We start from the **Lab 07 full-SFT checkpoint**, not the broken Lab 09 distilled checkpoint.

Why?

The starting policy must already produce recognizable Wordle-shaped output often enough for RL to obtain useful reward variation.

For the first GRPO experiment:

```text
starting policy:     Lab 07 full SFT
RL parameters:       LoRA adapters
group size:          4 generations
completion limit:    8 tokens
temperature:         0.9
training steps:      200
KL coefficient beta: 0.0
```

Using LoRA here is an implementation choice: it keeps the RL update small, protects the SFT checkpoint, and reduces memory/optimizer cost. The training method under study is still GRPO.

We explicitly use `beta=0.0`, which is also TRL's current GRPO default. That avoids loading a separate reference model. Lab 12 can ablate KL regularization if useful.

## 11.3 Restart the kernel and check versions

Install/update the RL packages if needed:

```bash
pip install -U trl peft accelerate
```

Then restart the Jupyter kernel.

In [3]:
from __future__ import annotations

from pathlib import Path
import json
import random
import re
import time

import numpy as np
import pandas as pd
import torch
import transformers
import trl
import peft

from datasets import Dataset, load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from tiny_wordle.game import Turn, filter_candidates, score_string
from tiny_wordle.expert import EntropyExpert
from tiny_wordle.rewards import reward_breakdown
from tiny_wordle.benchmark import (
    DEFAULT_SYSTEM_RULES,
    run_benchmark,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("device:", device)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

device: mps
PyTorch: 2.13.0
Transformers: 5.15.0
TRL: 1.10.0
PEFT: 0.20.0


TRL's current GRPO custom reward API passes:

- generated `completions`;
- the original `prompts`;
- every additional dataset column as a keyword argument.

That lets each training row carry the hidden answer and history as **environment metadata** without exposing them in the model prompt.

## 11.4 Load Wordle infrastructure

In [4]:
DATA_DIR = Path("../data")

ANSWERS = [
    line.strip().upper()
    for line in (DATA_DIR / "wordle-answers-original.txt").read_text().splitlines()
    if line.strip()
]

PATTERNS = np.load(
    DATA_DIR / "wordle-patterns-original-2315.npy"
)

expert = EntropyExpert(
    ANSWERS,
    PATTERNS,
)

WORD_TO_INDEX = expert.word_to_index
ALL_INDICES = expert.all_indices

print("answers:", len(ANSWERS))
print("pattern matrix:", PATTERNS.shape)

answers: 2315
pattern matrix: (2315, 2315)


## 11.5 Recover the same train/dev answer split as Lab 06

Rather than reimplementing split logic, read the persisted Lab 06 files and recover their answer sets.

In [5]:
GENERATED_DIR = DATA_DIR / "generated"

lab06 = load_dataset(
    "json",
    data_files={
        "train": str(GENERATED_DIR / "wordle-sft-train.jsonl"),
        "validation": str(GENERATED_DIR / "wordle-sft-dev.jsonl"),
    },
)

train_answers = sorted(
    set(lab06["train"]["answer"])
)

dev_answers = sorted(
    set(lab06["validation"]["answer"])
)

print("train answers:", len(train_answers))
print("dev answers:", len(dev_answers))

assert set(train_answers).isdisjoint(dev_answers)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

train answers: 2064
dev answers: 231


## 11.6 Build one-turn RL states

A training row contains a Wordle state:

```text
visible to model:
    gameplay prompt + history

hidden environment metadata:
    answer
    turn
    serialized history
```

We generate states by following the symbolic expert trajectory, but the **target action is not stored**.

GRPO must discover better actions from reward.

In [6]:
def format_history(history: list[Turn]) -> str:
    if not history:
        return "No guesses have been made yet."

    return "\n".join(
        f"{' '.join(t.guess)} -> {' '.join(t.feedback)}"
        for t in history
    )

def gameplay_prompt(history: list[Turn]) -> str:
    return (
        DEFAULT_SYSTEM_RULES
        + "\nGame history:\n"
        + format_history(history)
    )

def history_to_json(history: list[Turn]) -> str:
    return json.dumps([
        {
            "guess": t.guess,
            "feedback": t.feedback,
        }
        for t in history
    ])

def history_from_json(text: str) -> list[Turn]:
    return [
        Turn(
            item["guess"],
            item["feedback"],
        )
        for item in json.loads(text)
    ]

def collect_rl_states(
    answers: list[str],
    max_turns: int = 6,
) -> list[dict]:
    rows = []

    for answer in answers:
        candidate_indices = ALL_INDICES.copy()
        history: list[Turn] = []

        for turn in range(1, max_turns + 1):
            rows.append({
                "prompt": [
                    {
                        "role": "user",
                        "content": gameplay_prompt(history),
                    }
                ],
                "answer": answer,
                "turn": turn,
                "history_json": history_to_json(history),
                "candidate_count": int(len(candidate_indices)),
            })

            guess_idx = expert.choose(candidate_indices)
            guess = ANSWERS[guess_idx]
            feedback = score_string(answer, guess)

            if feedback == "GGGGG":
                break

            history.append(
                Turn(
                    guess,
                    feedback,
                )
            )

            candidate_indices = expert.update(
                candidate_indices,
                guess_idx,
                feedback,
            )

    return rows

In [7]:
all_train_states = collect_rl_states(
    train_answers
)

all_dev_states = collect_rl_states(
    dev_answers
)

print("train state pool:", len(all_train_states))
print("dev state pool:", len(all_dev_states))

train state pool: 7417
dev state pool: 833


## 11.7 Sample a practical first RL dataset

We do not need every state for the first GRPO run.

Use:

```text
1,200 training states
200 held-out diagnostic states
```

Stratification is intentionally simple; Lab 12 is where we can ablate state distributions.

In [14]:
RL_TRAIN_STATES = 1200
RL_DEV_STATES = 200

rng = random.Random(SEED)

train_rows = rng.sample(
    all_train_states,
    min(
        RL_TRAIN_STATES,
        len(all_train_states),
    ),
)

dev_rows = rng.sample(
    all_dev_states,
    min(
        RL_DEV_STATES,
        len(all_dev_states),
    ),
)

train_dataset = Dataset.from_list(
    train_rows
)

dev_dataset = Dataset.from_list(
    dev_rows
)

print(train_dataset)
print(dev_dataset)

print()
print("TRAIN TURN DISTRIBUTION")
print(
    pd.Series(
        train_dataset["turn"]
    ).value_counts().sort_index()
)

Dataset({
    features: ['prompt', 'answer', 'turn', 'history_json', 'candidate_count'],
    num_rows: 1200
})
Dataset({
    features: ['prompt', 'answer', 'turn', 'history_json', 'candidate_count'],
    num_rows: 200
})

TRAIN TURN DISTRIBUTION
1    345
2    352
3    280
4    178
5     37
6      8
Name: count, dtype: int64


## 11.8 Implement the GRPO reward adapter

Lab 10's reward function operates on structured environment facts.

TRL gives us generated text.

This wrapper connects them:

```text
completion
   ↓ parse guess
environment metadata
   ↓ score Wordle transition
Lab 10 reward_breakdown(...)
   ↓
scalar reward
```

The hidden answer is never added to the model prompt.

In [15]:
WORD_RE = re.compile(
    r"^[A-Za-z]{5}$"
)

def completion_text(completion) -> str:
    # Standard-format completion.
    if isinstance(completion, str):
        return completion.strip()

    # Conversational-format completion.
    if isinstance(completion, list):
        if not completion:
            return ""

        last = completion[-1]

        if isinstance(last, dict):
            return str(
                last.get(
                    "content",
                    "",
                )
            ).strip()

    return str(completion).strip()

def evaluate_completion(
    completion,
    *,
    answer: str,
    history_json: str,
    turn: int,
):
    raw = completion_text(
        completion
    )

    history = history_from_json(
        history_json
    )

    candidates_before = filter_candidates(
        ANSWERS,
        history,
    )

    valid_format = bool(
        WORD_RE.fullmatch(raw)
    )

    if not valid_format:
        breakdown = reward_breakdown(
            valid_format=False,
            repeated=False,
            history_consistent=False,
            has_history=bool(history),
            solved=False,
            candidates_before=len(candidates_before),
            candidates_after=len(candidates_before),
            turn_number=int(turn),
        )

        return {
            "raw": raw,
            "guess": None,
            "reward": breakdown.total,
            "breakdown": breakdown,
            "solved": False,
            "consistent": False,
            "repeated": False,
        }

    guess = raw.upper()

    if guess not in WORD_TO_INDEX:
        breakdown = reward_breakdown(
            valid_format=False,
            repeated=False,
            history_consistent=False,
            has_history=bool(history),
            solved=False,
            candidates_before=len(candidates_before),
            candidates_after=len(candidates_before),
            turn_number=int(turn),
        )

        return {
            "raw": raw,
            "guess": None,
            "reward": breakdown.total,
            "breakdown": breakdown,
            "solved": False,
            "consistent": False,
            "repeated": False,
        }

    repeated = any(
        old.guess == guess
        for old in history
    )

    consistent = all(
        score_string(
            guess,
            old.guess,
        ) == old.feedback
        for old in history
    )

    feedback = score_string(
        answer,
        guess,
    )

    solved = (
        feedback == "GGGGG"
    )

    new_history = history + [
        Turn(
            guess,
            feedback,
        )
    ]

    candidates_after = filter_candidates(
        ANSWERS,
        new_history,
    )

    breakdown = reward_breakdown(
        valid_format=True,
        repeated=repeated,
        history_consistent=consistent,
        has_history=bool(history),
        solved=solved,
        candidates_before=len(candidates_before),
        candidates_after=len(candidates_after),
        turn_number=int(turn),
    )

    return {
        "raw": raw,
        "guess": guess,
        "reward": breakdown.total,
        "breakdown": breakdown,
        "solved": solved,
        "consistent": consistent,
        "repeated": repeated,
    }

In [16]:
def wordle_reward(
    completions,
    answer,
    history_json,
    turn,
    **kwargs,
):
    rewards = []

    for completion, ans, hist, t in zip(
        completions,
        answer,
        history_json,
        turn,
    ):
        result = evaluate_completion(
            completion,
            answer=ans,
            history_json=hist,
            turn=t,
        )

        rewards.append(
            float(
                result["reward"]
            )
        )

    return rewards

## 11.9 Dry-run the reward adapter

Test several completions against one real environment state before giving the function to GRPO.

In [17]:
row = train_dataset[0]

print(
    row["prompt"][0]["content"]
)

print()
print("hidden answer:", row["answer"])
print("turn:", row["turn"])

for completion in [
    "CRANE",
    "SLATE",
    row["answer"],
    "xxxxx",
    "I think CRANE",
]:
    result = evaluate_completion(
        completion,
        answer=row["answer"],
        history_json=row["history_json"],
        turn=row["turn"],
    )

    print()
    print(
        repr(completion),
        "->",
        result,
    )

Play Wordle.

Return exactly one uppercase five-letter English word.
Do not explain.
Do not use punctuation.

Use all previous guesses and feedback when choosing the next guess.
Never repeat a previous guess.

Example valid response:
CRANE

Feedback meanings:
G = correct letter and position
Y = letter is present but wrong position
B = that letter occurrence is not matched

Game history:
R A I S E -> B B B Y Y

hidden answer: SCENT
turn: 2

'CRANE' -> {'raw': 'CRANE', 'guess': 'CRANE', 'reward': 0.09976218787158146, 'breakdown': RewardBreakdown(format=0.05, repeat=0.0, history=-0.15, progress=0.19976218787158145, solve=0.0, efficiency=0.0), 'solved': False, 'consistent': False, 'repeated': False}

'SLATE' -> {'raw': 'SLATE', 'guess': 'SLATE', 'reward': 0.08532951289398281, 'breakdown': RewardBreakdown(format=0.05, repeat=0.0, history=-0.15, progress=0.1853295128939828, solve=0.0, efficiency=0.0), 'solved': False, 'consistent': False, 'repeated': False}

'SCENT' -> {'raw': 'SCENT', 'gues

At this point, malformed text should be negative, a solve should receive a large positive reward, and useful legal guesses should generally receive small positive shaping reward.

Do not proceed if the wrapper disagrees with Lab 10.

## 11.10 Load the Lab 07 SFT checkpoint

This is our starting policy.

It previously achieved:

```text
valid output rate: 100%
solve rate:         5%
history consistency: ~0%
```

That is a reasonable RL starting point: the model speaks the interface, but its behavior leaves plenty of room for reward-driven improvement.

In [18]:
START_MODEL = (
    "../checkpoints/"
    "qwen3-0.6b-wordle-full-sft"
)

assert Path(START_MODEL).exists(), (
    f"Missing checkpoint: {START_MODEL}"
)

tokenizer = AutoTokenizer.from_pretrained(
    START_MODEL
)

tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    START_MODEL,
    dtype=torch.float32,
).to(device)

print(type(model).__name__)
print("device:", next(model.parameters()).device)
print("dtype:", next(model.parameters()).dtype)
print(
    "parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForCausalLM
device: mps:0
dtype: torch.float32
parameters: 596,049,920


## 11.11 Inspect pre-RL sampled actions

GRPO needs reward variance within a group.

Generate several guesses from the same prompt with sampling enabled and score them.

In [19]:
@torch.no_grad()
def sample_guesses(
    row,
    n: int = 8,
):
    chat_text = tokenizer.apply_chat_template(
        row["prompt"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    batch = tokenizer(
        chat_text,
        return_tensors="pt",
    ).to(device)

    outputs = model.generate(
        **batch,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        max_new_tokens=8,
        num_return_sequences=n,
    )

    records = []

    for output in outputs:
        new = output[
            batch["input_ids"].shape[1]:
        ]

        text = tokenizer.decode(
            new,
            skip_special_tokens=True,
        ).strip()

        result = evaluate_completion(
            text,
            answer=row["answer"],
            history_json=row["history_json"],
            turn=row["turn"],
        )

        records.append({
            "completion": text,
            "reward": result["reward"],
            "solved": result["solved"],
            "consistent": result["consistent"],
            "repeated": result["repeated"],
        })

    return pd.DataFrame(
        records
    )

sampled_before = sample_guesses(
    train_dataset[0],
    n=8,
)

sampled_before

,completion,reward,solved,consistent,repeated
0,CRANE NodeList,-0.250000,False,False,False
1,CRANE,0.099762,False,False,False
2,CRISE,-0.250000,False,False,False
3,CRANE,0.099762,False,False,False
4,CRANE,0.099762,False,False,False
5,CRANE,0.099762,False,False,False
6,REBIC,-0.250000,False,False,False
7,CRANE,0.099762,False,False,False


If every sample receives exactly the same reward, GRPO has no within-group learning signal for that prompt.

Some zero-variance groups are normal. If nearly all groups are identical, increase sampling temperature or reconsider the starting policy/reward.

## 11.12 Configure LoRA for the RL update

The SFT weights remain frozen.

GRPO updates only these attention adapters.

In [20]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    bias="none",
)

print(lora_config)

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'o_proj', 'k_proj', 'q_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## 11.13 Configure GRPO

Important parameters:

### `num_generations=4`

Four guesses are sampled for each prompt group. Their relative rewards determine the advantage.

### `temperature=0.9`

We need exploration. Greedy decoding would produce identical completions and therefore no group-relative signal.

### `max_completion_length=8`

The desired action is one five-letter word. Long completions are almost certainly bad.

### `beta=0.0`

Current TRL defaults to no reference-model KL term. This saves memory and makes the first run easier to interpret.

### `loss_type="dapo"`

Use TRL's modern token-normalized GRPO-style objective rather than the older length-biased formulation.

### `scale_rewards="group"`

Normalize rewards within each completion group—the core GRPO idea.

In [23]:
OUTPUT_DIR = (
    "../checkpoints/"
    "qwen3-0.6b-wordle-grpo-adapter"
)

MAX_STEPS = 200
NUM_GENERATIONS = 4
WARMUP_STEPS = max(1, int(MAX_STEPS * 0.05))

grpo_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    warmup_steps=WARMUP_STEPS,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,

    num_generations=NUM_GENERATIONS,
    max_completion_length=8,

    temperature=0.9,
    top_p=0.95,

    beta=0.0,
    scale_rewards="group",
    loss_type="dapo",

    chat_template_kwargs={
        "enable_thinking": False,
    },

    remove_unused_columns=False,

    logging_steps=5,
    logging_first_step=True,
    log_completions=True,
    num_completions_to_print=4,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    report_to="none",

    bf16=False,
    fp16=False,
    gradient_checkpointing=False,

    seed=SEED,
)

print(grpo_args)

GRPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.0,
bf16=False,
bf16_full_eval=False,
cache_implementation=None,
cast_lm_head_to_fp32=False,
chat_template_kwargs={'enable_thinking': False},
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
delta=None,
disable_dropout=False,
disable_tqdm=False,
do_eval=False,
do_pre

The effective batch size must be divisible by `num_generations`.

Here:

```text
1 process × batch 4 × accumulation 1 = 4
num_generations = 4
```

So each update starts from one prompt group with four sampled guesses.

In [24]:
effective_batch = (
    grpo_args.per_device_train_batch_size
    * grpo_args.gradient_accumulation_steps
)

assert (
    effective_batch
    % NUM_GENERATIONS
    == 0
)

print(
    "effective batch:",
    effective_batch,
)
print(
    "generations/group:",
    NUM_GENERATIONS,
)

effective batch: 4
generations/group: 4


## 11.14 Construct the GRPO trainer

The trainer will:

1. sample prompts from `train_dataset`;
2. generate four guesses from the current policy;
3. call `wordle_reward`;
4. normalize rewards within the group;
5. update the LoRA policy toward higher-advantage completions.

In [25]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=wordle_reward,
    args=grpo_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


The trainable parameter count should be close to Lab 08's LoRA count, not 596M.

That is expected: GRPO is changing the **optimization objective**, while LoRA controls which parameters are allowed to move.

## 11.15 Train

This is the first online RL run in the course.

Watch:

- reward mean;
- reward standard deviation;
- fraction of zero-variance groups;
- completion length;
- entropy;
- clipping statistics;
- malformed completions.

A decreasing "loss" is not the main success criterion in RL. Reward and downstream behavior matter more.

In [26]:
start = time.perf_counter()

train_result = trainer.train()

elapsed = (
    time.perf_counter()
    - start
)

print()
print(
    "GRPO training seconds:",
    elapsed,
)

train_result

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/ianphil/src/tiny-wordle-lab-v2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.199621
5,-0.002818
10,0.047162
15,0.011690
20,0.016639
25,0.054441
30,0.000000
35,-0.015129
40,-0.006571
45,0.049700


╭──────────────────────────────────────────────────── Step 1 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                             ┃ Completion ┃ wordle_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                                               │ EMERG      │         -0.25 │     -0.87 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Return exactly one uppercase five-letter English word.             │            │               │           │ │
│ │ Do not explain.                                                    │            │               │           │ │
│ │ Do not use punctuation.                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Use all previous guesses and feedback when choosing the next       │            │               │           │ │
│ │ guess.                                                             │            │               │           │ │
│ │ Never repeat a previous guess.                                     │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Example valid response:                                            │            │               │           │ │
│ │ CRANE                                                              │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Feedback meanings:                                                 │            │               │           │ │
│ │ G = correct letter and position                                    │            │               │           │ │
│ │ Y = letter is present but wrong position                           │            │               │           │ │
│ │ B = that letter occurrence is not matched                          │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Game history:                                                      │            │               │           │ │
│ │ R A I S E -> Y B B B Y                                             │            │               │           │ │
│ │ D E T E R -> B B B G Y                                             │            │               │           │ │
│ │ assistant                                                          │            │               │           │ │
│ │ <think>                                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ </think>                                                           │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ ├────────────────────────────────────────────────────────────────────┼────────────┼───────────────┼───────────┤ │
│ │ user                                                               │ GLERMY     │         -0.25 │     -0.87 │ │
│ │ Play Wordle.                                        

╭──────────────────────────────────────────────────── Step 5 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                             ┃ Completion ┃ wordle_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                                               │ CRANE      │         -0.01 │      0.23 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Return exactly one uppercase five-letter English word.             │            │               │           │ │
│ │ Do not explain.                                                    │            │               │           │ │
│ │ Do not use punctuation.                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Use all previous guesses and feedback when choosing the next       │            │               │           │ │
│ │ guess.                                                             │            │               │           │ │
│ │ Never repeat a previous guess.                                     │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Example valid response:                                            │            │               │           │ │
│ │ CRANE                                                              │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Feedback meanings:                                                 │            │               │           │ │
│ │ G = correct letter and position                                    │            │               │           │ │
│ │ Y = letter is present but wrong position                           │            │               │           │ │
│ │ B = that letter occurrence is not matched                          │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Game history:                                                      │            │               │           │ │
│ │ R A I S E -> B B B B G                                             │            │               │           │ │
│ │ assistant                                                          │            │               │           │ │
│ │ <think>                                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ </think>                                                           │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ ├────────────────────────────────────────────────────────────────────┼────────────┼───────────────┼───────────┤ │
│ │ user                                                               │ BRIDE      │          0.09 │      0.96 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                     

╭──────────────────────────────────────────────────── Step 10 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                             ┃ Completion ┃ wordle_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                                               │ CRANE      │          0.09 │      0.66 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Return exactly one uppercase five-letter English word.             │            │               │           │ │
│ │ Do not explain.                                                    │            │               │           │ │
│ │ Do not use punctuation.                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Use all previous guesses and feedback when choosing the next       │            │               │           │ │
│ │ guess.                                                             │            │               │           │ │
│ │ Never repeat a previous guess.                                     │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Example valid response:                                            │            │               │           │ │
│ │ CRANE                                                              │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Feedback meanings:                                                 │            │               │           │ │
│ │ G = correct letter and position                                    │            │               │           │ │
│ │ Y = letter is present but wrong position                           │            │               │           │ │
│ │ B = that letter occurrence is not matched                          │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Game history:                                                      │            │               │           │ │
│ │ R A I S E -> B B B Y B                                             │            │               │           │ │
│ │ assistant                                                          │            │               │           │ │
│ │ <think>                                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ </think>                                                           │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ ├────────────────────────────────────────────────────────────────────┼────────────┼───────────────┼───────────┤ │
│ │ user                                                               │ SCREW      │          0.00 │      0.12 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                     

...

╭─────────────────────────────────────────────────── Step 195 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                             ┃ Completion ┃ wordle_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                                               │ CRANE      │          0.25 │     -0.24 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Return exactly one uppercase five-letter English word.             │            │               │           │ │
│ │ Do not explain.                                                    │            │               │           │ │
│ │ Do not use punctuation.                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Use all previous guesses and feedback when choosing the next       │            │               │           │ │
│ │ guess.                                                             │            │               │           │ │
│ │ Never repeat a previous guess.                                     │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Example valid response:                                            │            │               │           │ │
│ │ CRANE                                                              │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Feedback meanings:                                                 │            │               │           │ │
│ │ G = correct letter and position                                    │            │               │           │ │
│ │ Y = letter is present but wrong position                           │            │               │           │ │
│ │ B = that letter occurrence is not matched                          │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Game history:                                                      │            │               │           │ │
│ │ No guesses have been made yet.                                     │            │               │           │ │
│ │ assistant                                                          │            │               │           │ │
│ │ <think>                                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ </think>                                                           │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ ├────────────────────────────────────────────────────────────────────┼────────────┼───────────────┼───────────┤ │
│ │ user                                                               │ CRANE      │          0.25 │     -0.24 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                     

╭─────────────────────────────────────────────────── Step 200 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                             ┃ Completion ┃ wordle_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                                               │ TANGAN     │         -0.25 │     -0.50 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Return exactly one uppercase five-letter English word.             │            │               │           │ │
│ │ Do not explain.                                                    │            │               │           │ │
│ │ Do not use punctuation.                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Use all previous guesses and feedback when choosing the next       │            │               │           │ │
│ │ guess.                                                             │            │               │           │ │
│ │ Never repeat a previous guess.                                     │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Example valid response:                                            │            │               │           │ │
│ │ CRANE                                                              │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Feedback meanings:                                                 │            │               │           │ │
│ │ G = correct letter and position                                    │            │               │           │ │
│ │ Y = letter is present but wrong position                           │            │               │           │ │
│ │ B = that letter occurrence is not matched                          │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Game history:                                                      │            │               │           │ │
│ │ R A I S E -> B G B B B                                             │            │               │           │ │
│ │ T A N G Y -> Y G B B B                                             │            │               │           │ │
│ │ assistant                                                          │            │               │           │ │
│ │ <think>                                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ </think>                                                           │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ ├────────────────────────────────────────────────────────────────────┼────────────┼───────────────┼───────────┤ │
│ │ user                                                               │ SABLY      │         -0.25 │     -0.50 │ │
│ │ Play Wordle.                                        

╭─────────────────────────────────────────────────── Step 200 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                             ┃ Completion ┃ wordle_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                                               │ TANGAN     │         -0.25 │     -0.50 │ │
│ │ Play Wordle.                                                       │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Return exactly one uppercase five-letter English word.             │            │               │           │ │
│ │ Do not explain.                                                    │            │               │           │ │
│ │ Do not use punctuation.                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Use all previous guesses and feedback when choosing the next       │            │               │           │ │
│ │ guess.                                                             │            │               │           │ │
│ │ Never repeat a previous guess.                                     │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Example valid response:                                            │            │               │           │ │
│ │ CRANE                                                              │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Feedback meanings:                                                 │            │               │           │ │
│ │ G = correct letter and position                                    │            │               │           │ │
│ │ Y = letter is present but wrong position                           │            │               │           │ │
│ │ B = that letter occurrence is not matched                          │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ Game history:                                                      │            │               │           │ │
│ │ R A I S E -> B G B B B                                             │            │               │           │ │
│ │ T A N G Y -> Y G B B B                                             │            │               │           │ │
│ │ assistant                                                          │            │               │           │ │
│ │ <think>                                                            │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │ </think>                                                           │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ │                                                                    │            │               │           │ │
│ ├────────────────────────────────────────────────────────────────────┼────────────┼───────────────┼───────────┤ │
│ │ user                                                               │ SABLY      │         -0.25 │     -0.50 │ │
│ │ Play Wordle.                                        


GRPO training seconds: 76.03762833399924


TrainOutput(global_step=200, training_loss=0.019403464334706477, metrics={'train_runtime': 75.8593, 'train_samples_per_second': 10.546, 'train_steps_per_second': 2.636, 'total_flos': 0.0, 'train_loss': 0.019403464334706477, 'epoch': 0.16666666666666666})

## 11.16 Inspect GRPO logs

TRL stores the trainer log history.

Column names can evolve between TRL releases, so inspect what was actually logged rather than hard-coding assumptions.

In [27]:
log_df = pd.DataFrame(
    trainer.state.log_history
)

print(
    "log columns:"
)

for column in log_df.columns:
    print(" -", column)

log_df.tail(20)

log columns:
 - loss
 - grad_norm
 - learning_rate
 - num_tokens
 - completions/mean_length
 - completions/min_length
 - completions/max_length
 - completions/clipped_ratio
 - completions/mean_terminated_length
 - completions/min_terminated_length
 - completions/max_terminated_length
 - rewards/wordle_reward/mean
 - rewards/wordle_reward/std
 - reward
 - reward_std
 - frac_reward_zero_std
 - entropy
 - clip_ratio/low_mean
 - clip_ratio/high_mean
 - clip_ratio/region_mean
 - clip_ratio/low_min
 - clip_ratio/high_max
 - step_time
 - epoch
 - step
 - train_runtime
 - train_samples_per_second
 - train_steps_per_second
 - total_flos
 - train_loss


,loss,grad_norm,learning_rate,num_tokens,completions/mean_length,completions/min_length,completions/max_length,completions/clipped_ratio,completions/mean_terminated_length,completions/min_terminated_length,...,clip_ratio/low_min,clip_ratio/high_max,step_time,epoch,step,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
22,0.004265,1.408403,2.394737e-06,48191.0,3.25,3.0,3.6,0.0,3.25,3.0,...,0.0,0.0,0.301410,0.091667,110,NaN,NaN,NaN,NaN,NaN
23,0.060300,2.240368,2.263158e-06,50390.0,3.35,2.8,4.0,0.0,3.35,2.8,...,0.0,0.0,0.306911,0.095833,115,NaN,NaN,NaN,NaN,NaN
24,-0.004743,1.717145,2.131579e-06,52623.0,3.45,3.2,3.6,0.0,3.45,3.2,...,0.0,0.0,0.306965,0.100000,120,NaN,NaN,NaN,NaN,NaN
25,0.038103,2.387602,2.000000e-06,54835.0,3.20,2.8,3.8,0.0,3.20,2.8,...,0.0,0.0,0.322863,0.104167,125,NaN,NaN,NaN,NaN,NaN
26,-0.016028,1.783125,1.868421e-06,56983.0,3.20,3.0,3.4,0.0,3.20,3.0,...,0.0,0.0,0.292639,0.108333,130,NaN,NaN,NaN,NaN,NaN
27,0.006246,2.949647,1.736842e-06,59129.0,3.10,2.6,3.4,0.0,3.10,2.6,...,0.0,0.0,0.290910,0.112500,135,NaN,NaN,NaN,NaN,NaN
28,0.022872,2.921869,1.605263e-06,61212.0,3.15,2.8,3.4,0.0,3.15,2.8,...,0.0,0.0,0.286668,0.116667,140,NaN,NaN,NaN,NaN,NaN
29,0.038634,3.634436,1.473684e-06,63469.0,3.05,2.6,3.4,0.0,3.05,2.6,...,0.0,0.0,0.297531,0.120833,145,NaN,NaN,NaN,NaN,NaN
30,0.043082,2.546692,1.342105e-06,65841.0,3.20,2.8,3.6,0.0,3.20,2.8,...,0.0,0.0,0.355581,0.125000,150,NaN,NaN,NaN,NaN,NaN
31,-0.013702,1.858616,1.210526e-06,68037.0,3.20,3.0,3.6,0.0,3.20,3.0,...,0.0,0.0,0.302125,0.129167,155,NaN,NaN,NaN,NaN,NaN


In [29]:
rl_metrics = log_df[
    log_df["reward"].notna()
][[
    "step",
    "loss",
    "grad_norm",
    "learning_rate",
    "reward",
    "reward_std",
    "frac_reward_zero_std",
    "entropy",
    "completions/mean_length",
    "completions/clipped_ratio",
]]

rl_metrics.to_string(index=False)

' step          loss  grad_norm  learning_rate    reward  reward_std  frac_reward_zero_std  entropy  completions/mean_length  completions/clipped_ratio\n    1  1.996214e-01   1.368716   0.000000e+00 -0.175000    0.086603                   0.0 2.097321                    3.250                        0.0\n    5 -2.817898e-03   3.248951   2.000000e-06 -0.046851    0.172897                   0.0 2.132856                    3.125                        0.0\n   10  4.716229e-02   3.315003   4.500000e-06  0.414706    0.103230                   0.4 1.632640                    3.050                        0.0\n   15  1.169036e-02   3.500400   4.894737e-06 -0.129889    0.134706                   0.0 1.843699                    3.450                        0.0\n   20  1.663933e-02   2.248376   4.763158e-06 -0.049778    0.098115                   0.0 1.894456                    3.300                        0.0\n   25  5.444067e-02   2.268942   4.631579e-06  0.056634    0.224331                   0

In [30]:
print(
    rl_metrics[
        [
            "reward",
            "reward_std",
            "frac_reward_zero_std",
            "entropy",
            "completions/mean_length",
            "completions/clipped_ratio",
        ]
    ].describe()
)

          reward  reward_std  frac_reward_zero_std    entropy  \
count  41.000000   41.000000             41.000000  41.000000   
mean    0.012762    0.198020              0.078049   1.644695   
std     0.108580    0.126379              0.125523   0.224588   
min    -0.175000    0.055244              0.000000   1.113812   
25%    -0.054151    0.115296              0.000000   1.494253   
50%    -0.001602    0.148267              0.000000   1.632421   
75%     0.056634    0.224331              0.200000   1.819766   
max     0.414706    0.642321              0.400000   2.132856   

       completions/mean_length  completions/clipped_ratio  
count                41.000000                       41.0  
mean                  3.193293                        0.0  
std                   0.140426                        0.0  
min                   3.000000                        0.0  
25%                   3.100000                        0.0  
50%                   3.200000                        

In [28]:
interesting = [
    c
    for c in log_df.columns
    if any(
        key in c.lower()
        for key in [
            "reward",
            "loss",
            "entropy",
            "clip",
            "completion",
            "grad",
        ]
    )
]

print(
    log_df[
        interesting
    ].tail(30)
)

        loss  grad_norm  completions/mean_length  completions/min_length  \
12  0.022621   3.289606                     3.25                     3.0   
13 -0.013277   2.601483                     3.40                     3.0   
14  0.022954   3.165412                     3.25                     2.8   
15  0.070835   1.554769                     3.20                     3.0   
16  0.029608   1.574702                     3.35                     3.0   
17 -0.035491   2.528873                     3.35                     3.0   
18  0.001021   0.000000                     3.05                     2.6   
19 -0.000699   1.814368                     3.05                     3.0   
20 -0.010220   0.924099                     3.25                     3.0   
21  0.015013   0.000000                     3.00                     2.8   
22  0.004265   1.408403                     3.25                     3.0   
23  0.060300   2.240368                     3.35                     2.8   
24 -0.004743

Look especially for evidence that the group has reward variance.

If `frac_reward_zero_std` is near 1.0, GRPO is frequently seeing groups where all four completions score the same, so there is little relative learning signal.

## 11.17 Save the final adapter

In [31]:
FINAL_ADAPTER_DIR = Path(
    "../checkpoints/"
    "qwen3-0.6b-wordle-grpo-final-adapter"
)

trainer.model.save_pretrained(
    FINAL_ADAPTER_DIR
)

tokenizer.save_pretrained(
    FINAL_ADAPTER_DIR
)

print(
    "saved adapter:",
    FINAL_ADAPTER_DIR,
)

saved adapter: ../checkpoints/qwen3-0.6b-wordle-grpo-final-adapter


## 11.18 Merge the RL adapter for easy benchmarking

Our reusable benchmark loads ordinary standalone causal-LM checkpoints.

Merge:

```text
Lab 07 SFT weights
      +
Lab 11 GRPO LoRA adapter
      ↓
standalone RL checkpoint
```

In [32]:
MERGED_DIR = Path(
    "../checkpoints/"
    "qwen3-0.6b-wordle-grpo-merged"
)

merged_model = (
    trainer.model
    .merge_and_unload()
)

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    MERGED_DIR
)

print(
    "merged checkpoint:",
    MERGED_DIR,
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

merged checkpoint: ../checkpoints/qwen3-0.6b-wordle-grpo-merged


## 11.19 Free training memory before gameplay evaluation

In [33]:
del trainer
del model

if device.type == "mps":
    torch.mps.empty_cache()

print("training objects released")

training objects released


## 11.20 Run the reusable 19-game benchmark

No 20-cell rerun.

One call.

In [34]:
summary, games_df, results = run_benchmark(
    str(MERGED_DIR),
    verbose_games=True,
    print_failures=False,
)

summary

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

model: ../checkpoints/qwen3-0.6b-wordle-grpo-merged
device: mps
dtype: torch.float32
games: 19

 1/19 SHORE FAILED turns=6 valid=6 invalid=0 repeats=5
 2/19 MIGHT FAILED turns=6 valid=6 invalid=0 repeats=4
 3/19 BRICK FAILED turns=6 valid=6 invalid=0 repeats=4
 4/19 GHOST FAILED turns=6 valid=6 invalid=0 repeats=4
 5/19 KNIFE FAILED turns=6 valid=6 invalid=0 repeats=5
 6/19 DOUBT FAILED turns=6 valid=6 invalid=0 repeats=4
 7/19 FLING FAILED turns=6 valid=6 invalid=0 repeats=5
 8/19 ROUND FAILED turns=6 valid=6 invalid=0 repeats=5
 9/19 CHAMP FAILED turns=6 valid=6 invalid=0 repeats=5
10/19 WASTE FAILED turns=6 valid=6 invalid=0 repeats=5
11/19 BLIND FAILED turns=6 valid=6 invalid=0 repeats=5
12/19 POINT FAILED turns=6 valid=6 invalid=0 repeats=5
13/19 SLATE FAILED turns=6 valid=6 invalid=0 repeats=5
14/19 CRANE SOLVED turns=1 valid=1 invalid=0 repeats=0
15/19 APPLE FAILED turns=6 valid=6 invalid=0 repeats=5
16/19 SHEEP FAILED turns=6 valid=6 invalid=0 repeats=5
17/19 BANAL FAILED turns

{'model_id': '../checkpoints/qwen3-0.6b-wordle-grpo-merged',
 'games': 19,
 'solved': 1,
 'solve_rate': 0.05263157894736842,
 'model_calls': 109,
 'valid_output_rate': 1.0,
 'invalid_output_rate': 0.0,
 'repeat_guesses': 86,
 'history_consistent_guesses': 0,
 'history_consistency_checked': 90,
 'history_consistency_rate': 0.0,
 'mean_turns_on_wins': 1.0,
 'total_eval_seconds': 7.104556414989929}

## 11.21 Compare before and after RL

The Lab 07 starting checkpoint gave roughly:

```text
solve rate:           5%
valid output rate:  100%
history consistency:  0%
repetition failures: dominant
```

Fill in the GRPO result:

| Metric | Pre-RL SFT | GRPO |
|---|---:|---:|
| Solve rate | 5.0% | ? |
| Valid output rate | 100% | ? |
| History consistency | 0% | ? |
| Repeat guesses | high | ? |
| Mean turns on wins | 1.0 | ? |

The purpose of Lab 11 is not "GRPO must beat SFT."

The lesson is whether an online reward-driven update changes behavior in the direction specified by Lab 10.

## 11.22 Inspect one failed and one successful trajectory

Reward curves can hide bizarre behavior.

Always inspect actual generations.

In [35]:
solved_games = [
    r for r in results
    if r.solved
]

failed_games = [
    r for r in results
    if not r.solved
]

if solved_games:
    print("SOLVED EXAMPLE")
    print("answer:", solved_games[0].answer)
    for step in solved_games[0].trace:
        print(step)

print()

if failed_games:
    print("FAILED EXAMPLE")
    print("answer:", failed_games[0].answer)
    for step in failed_games[0].trace:
        print(step)

SOLVED EXAMPLE
answer: CRANE
{'turn': 1, 'raw_output': 'CRANE', 'guess': 'CRANE', 'feedback': 'GGGGG', 'status': 'valid_guess'}

FAILED EXAMPLE
answer: SHORE
{'turn': 1, 'raw_output': 'CRANE', 'guess': 'CRANE', 'feedback': 'BYBBG', 'status': 'valid_guess'}
{'turn': 2, 'raw_output': 'CRANE', 'guess': 'CRANE', 'feedback': 'BYBBG', 'status': 'repeat_guess'}
{'turn': 3, 'raw_output': 'CRANE', 'guess': 'CRANE', 'feedback': 'BYBBG', 'status': 'repeat_guess'}
{'turn': 4, 'raw_output': 'CRANE', 'guess': 'CRANE', 'feedback': 'BYBBG', 'status': 'repeat_guess'}
{'turn': 5, 'raw_output': 'CRANE', 'guess': 'CRANE', 'feedback': 'BYBBG', 'status': 'repeat_guess'}
{'turn': 6, 'raw_output': 'CRANE', 'guess': 'CRANE', 'feedback': 'BYBBG', 'status': 'repeat_guess'}


## 11.23 What did GRPO actually optimize?

GRPO never saw the symbolic expert's preferred action.

It only saw scalar consequences:

```text
valid format
repeat penalty
history consistency
candidate reduction
solve bonus
turn efficiency
```

That is the central contrast with all previous labs.

### Lab 07 — SFT

```text
expert says: "output FLOAT"
```

### Lab 09 — distillation

```text
expert says:
FLOAT .4
PLANT .2
...
```

### Lab 11 — RL

```text
model tries something
environment says: +0.37
```

The policy must infer which generated behavior caused the reward.

## 11.24 Important limitation of this first RL run

This is **one-step RL over Wordle states**, not end-to-end six-turn episodic GRPO.

Each rollout generates one guess and receives one shaped reward.

Why start here?

Because it lets us learn:

- online sampling;
- group-relative advantages;
- custom rewards;
- exploration;
- policy updates;
- reward variance;
- reward hacking;

without simultaneously building a stateful multi-turn agent environment.

A later extension could use TRL's environment interface for full multi-turn episodes. Current TRL also supports environment-backed GRPO, but that adds another layer of machinery and is not necessary for learning the core algorithm in this course.

# Lab 11 checkpoint

Send me these results in order:

1. package versions;
2. train/dev state-pool sizes and turn distribution;
3. the reward-wrapper dry run;
4. the eight pre-RL sampled guesses and rewards;
5. GRPO trainable parameter count;
6. first ~20 training steps/logs;
7. reward/zero-variance/entropy metrics from `log_history`;
8. total training time;
9. the final 19-game benchmark summary;
10. one solved and one failed trajectory if available.

Then Lab 12 will run a small set of **ablations** and produce the final report comparing:

```text
base
full SFT
LoRA
distillation
RL
```